<a href="https://colab.research.google.com/github/SH1kha1/reWear/blob/main/Smart_Personal_AI_Assistant_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q \
langgraph \
langchain \
langchain-community \
langchain-openai \
langchain-google-genai \
langsmith \
google-api-python-client \
google-auth \
google-auth-oauthlib \
google-auth-httplib2 \
faiss-cpu \
pypdf \
tiktoken

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 120.4/120.4 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.7/70.7 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 50.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 56.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [ ]:
import langgraph
import langchain
import langsmith

print("✅ All packages imported successfully!")

✅ All packages imported successfully!


In [ ]:
import os

folders = [
    "data",
    "credentials",
    "vector_db",
    "outputs"
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

print("Project folders created successfully!")

Project folders created successfully!


In [ ]:
from google.colab import userdata
import os

os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
os.environ["LANGCHAIN_TRACING_V2"] = "true"

os.environ["LANGCHAIN_PROJECT"] = "Smart Personal Assistant"


In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

# Initialize the Gemini model
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)


# Test it
response = llm.invoke("Hello! Tell me who you are in one sentence.")

print(response.content)

I am a large language model, trained by Google.


In [ ]:
import shutil

# Find the uploaded JSON file
for file in os.listdir():
    if file.endswith(".json"):
        shutil.move(file, "credentials//content/credentials/client_secret_413644123909-mssrd09lu6esrabekit8156mmfg8tv63.apps.googleusercontent.com.json")
        break

print("✅ Credentials saved to credentials")

✅ Credentials saved to credentials


In [ ]:
!pip install --quiet google-api-python-client

In [ ]:
from googleapiclient.discovery import build

calendar_service = build(
    "calendar",
    "v3",
    credentials=creds
)

print("✅ Connected to Google Calendar")

✅ Connected to Google Calendar


In [ ]:
import os

print(os.listdir("credentials"))

['client_secret_413644123909-mssrd09lu6esrabekit8156mmfg8tv63.apps.googleusercontent.com.json', 'token.json', '.ipynb_checkpoints']


In [ ]:
from google.oauth2.credentials import Credentials

SCOPES = [
    "https://www.googleapis.com/auth/calendar",
    "https://www.googleapis.com/auth/gmail.modify"
]

creds = Credentials.from_authorized_user_file(
    "credentials/token.json",
    SCOPES
)

print("✅ Credentials loaded successfully!")
print("Valid:", creds.valid)

✅ Credentials loaded successfully!
Valid: True


In [ ]:
from googleapiclient.discovery import build

calendar_service = build(
    "calendar",
    "v3",
    credentials=creds
)

print("✅ Connected to Google Calendar!")

✅ Connected to Google Calendar!


List Today's Events

In [ ]:
from datetime import datetime, timedelta, timezone

def list_today_events():
    """
    Returns all events scheduled for today
    from the user's primary Google Calendar.
    """

    try:
        # Current UTC time
        now = datetime.now(timezone.utc)

        # End of the next 24 hours
        end = now + timedelta(days=1)

        events_result = calendar_service.events().list(
            calendarId="primary",
            timeMin=now.isoformat(),
            timeMax=end.isoformat(),
            singleEvents=True,
            orderBy="startTime"
        ).execute()

        events = events_result.get("items", [])

        if not events:
            return "📅 No events found for today."

        output = []

        for event in events:
            start = event["start"].get(
                "dateTime",
                event["start"].get("date")
            )

            title = event.get("summary", "Untitled Event")

            output.append(
                f"📅 {title}\n"
                f"🕒 {start}\n"
            )

        return "\n".join(output)

    except Exception as e:
        return f"❌ Error: {e}"

In [ ]:
print(list_today_events())

📅 No events found for today.


Create Calendar Event

In [ ]:
from datetime import datetime, timedelta

def create_calendar_event(
    title,
    start_datetime,
    end_datetime=None,
    description="",
    location=""
):
    """
    Creates a new event in the user's Google Calendar.

    If end_datetime is not provided,
    the meeting duration defaults to 1 hour.
    """

    try:

        # Automatically make the meeting 1 hour long
        if end_datetime is None:
            start = datetime.fromisoformat(start_datetime)
            end = start + timedelta(hours=1)
            end_datetime = end.isoformat()

        event = {
            "summary": title,
            "location": location,
            "description": description,
            "start": {
                "dateTime": start_datetime,
                "timeZone": "Asia/Riyadh"
            },
            "end": {
                "dateTime": end_datetime,
                "timeZone": "Asia/Riyadh"
            }
        }

        created_event = calendar_service.events().insert(
            calendarId="primary",
            body=event
        ).execute()

        return {
            "status": "success",
            "message": "Event created successfully.",
            "event_id": created_event["id"],
            "link": created_event["htmlLink"]
        }

    except Exception as e:
        return {
            "status": "error",
            "message": str(e)
        }

In [ ]:
result = create_calendar_event(
    title="AI Capstone Meeting",
    start_datetime="2026-07-10T14:00:00",
    end_datetime="2026-07-10T15:00:00",
    description="Testing my Smart Personal Assistant",
    location="Online"
)

print(result)

{'status': 'success', 'message': 'Event created successfully.', 'event_id': 'jto62q1geacv452jo0d5skug10', 'link': 'https://www.google.com/calendar/event?eid=anRvNjJxMWdlYWN2NDUyam8wZDVza3VnMTAgZGFsYWxhYmR1bGxhdGlmNjJAbQ'}


Search Calendar Events



In [ ]:
from datetime import datetime, timedelta, timezone

def search_calendar_events(keyword, days=30):
    """
    Search for calendar events containing a keyword.
    """

    try:
        now = datetime.now(timezone.utc)
        end = now + timedelta(days=days)

        events_result = calendar_service.events().list(
            calendarId="primary",
            timeMin=now.isoformat(),
            timeMax=end.isoformat(),
            q=keyword,
            singleEvents=True,
            orderBy="startTime"
        ).execute()

        events = events_result.get("items", [])

        if not events:
            return {
                "status": "success",
                "count": 0,
                "events": []
            }

        result = []

        for event in events:
            result.append({
                "id": event["id"],
                "title": event.get("summary", "Untitled"),
                "start": event["start"].get(
                    "dateTime",
                    event["start"].get("date")
                ),
                "link": event.get("htmlLink", "")
            })

        return {
            "status": "success",
            "count": len(result),
            "events": result
        }

    except Exception as e:
        return {
            "status": "error",
            "message": str(e)
        }

In [ ]:
result = search_calendar_events("AI")

print(result)

{'status': 'success', 'count': 1, 'events': [{'id': 's1jbn991mcgk8sg0g865rgejmg', 'title': 'AI Capstone Meeting', 'start': '2026-07-10T14:00:00+03:00', 'link': 'https://www.google.com/calendar/event?eid=czFqYm45OTFtY2drOHNnMGc4NjVyZ2VqbWcgZGFsYWxhYmR1bGxhdGlmNjJAbQ'}]}


Update Calendar Event

In [ ]:
def update_calendar_event(
    event_id,
    title=None,
    start_datetime=None,
    end_datetime=None,
    description=None,
    location=None
):
    """
    Update an existing Google Calendar event.
    """

    try:
        event = calendar_service.events().get(
            calendarId="primary",
            eventId=event_id
        ).execute()

        if title is not None:
            event["summary"] = title

        if description is not None:
            event["description"] = description

        if location is not None:
            event["location"] = location

        if start_datetime is not None:
            event["start"] = {
                "dateTime": start_datetime,
                "timeZone": "Asia/Riyadh"
            }

        if end_datetime is not None:
            event["end"] = {
                "dateTime": end_datetime,
                "timeZone": "Asia/Riyadh"
            }

        updated_event = calendar_service.events().update(
            calendarId="primary",
            eventId=event_id,
            body=event
        ).execute()

        return {
            "status": "success",
            "message": "Event updated successfully.",
            "event_id": updated_event["id"],
            "link": updated_event["htmlLink"]
        }

    except Exception as e:
        return {
            "status": "error",
            "message": str(e)
        }

In [ ]:
result = search_calendar_events("AI")

event_id = result["events"][0]["id"]

update_calendar_event(
    event_id=event_id,
    title="AI Final Project Meeting",
    description="Updated by my Smart Personal Assistant"
)

{'status': 'success',
 'message': 'Event updated successfully.',
 'event_id': 's1jbn991mcgk8sg0g865rgejmg',
 'link': 'https://www.google.com/calendar/event?eid=czFqYm45OTFtY2drOHNnMGc4NjVyZ2VqbWcgZGFsYWxhYmR1bGxhdGlmNjJAbQ'}

Delete Event

In [ ]:
def delete_calendar_event(event_id):
    """
    Delete an event from Google Calendar.
    """

    try:
        calendar_service.events().delete(
            calendarId="primary",
            eventId=event_id
        ).execute()

        return {
            "status": "success",
            "message": "Event deleted successfully."
        }

    except Exception as e:
        return {
            "status": "error",
            "message": str(e)
        }

In [ ]:
result = search_calendar_events("AI")

event_id = result["events"][0]["id"]

delete_calendar_event(event_id)

{'status': 'success', 'message': 'Event deleted successfully.'}

Create LangChain Tools

In [ ]:
from langchain_core.tools import tool

In [ ]:
@tool
def list_today_events_tool():
    """
    List today's events from Google Calendar.
    """
    return list_today_events()

In [ ]:
@tool
def create_calendar_event_tool(
    title: str,
    start_datetime: str,
    end_datetime: str,
    description: str = "",
    location: str = ""
):
    """
    Create a Google Calendar event.
    """
    return create_calendar_event(
        title,
        start_datetime,
        end_datetime,
        description,
        location
    )

In [ ]:
@tool
def search_calendar_events_tool(
    keyword: str
):
    """
    Search calendar events.
    """
    return search_calendar_events(keyword)

In [ ]:
@tool
def update_calendar_event_tool(
    event_id: str,
    title: str = None,
    start_datetime: str = None,
    end_datetime: str = None,
    description: str = None,
    location: str = None
):
    """
    Update a calendar event.
    """
    return update_calendar_event(
        event_id,
        title,
        start_datetime,
        end_datetime,
        description,
        location
    )

In [ ]:
@tool
def delete_calendar_event_tool(
    event_id: str
):
    """
    Delete a calendar event.
    """
    return delete_calendar_event(event_id)

In [ ]:
calendar_tools = [
    list_today_events_tool,
    create_calendar_event_tool,
    search_calendar_events_tool,
    update_calendar_event_tool,
    delete_calendar_event_tool
]

# ==========================================
# Module 4 - Calendar Agent
# ==========================================

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.prebuilt import create_react_agent

In [ ]:
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)

In [ ]:
calendar_agent = create_react_agent(
    model=llm,
    tools=calendar_tools,
    name="calendar_agent",
    prompt="""
You are an expert Google Calendar assistant.

Your responsibilities:

- List calendar events
- Create events
- Update events
- Delete events
- Search events

Always use the available tools.

Never make up information.

If information is missing, ask the user.

Today's timezone is Asia/Riyadh.
"""
)

/tmp/ipykernel_891/2226860260.py:1: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  calendar_agent = create_react_agent(


In [ ]:
response = calendar_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "What events do I have today?"
            }
        ]
    }
)

print(response["messages"][-1].content)

You have no events today.


# ==========================================
# Module 5 - Gmail Tools
# ==========================================

In [ ]:
from googleapiclient.discovery import build

gmail_service = build(
    "gmail",
    "v1",
    credentials=creds
)

print("✅ Connected to Gmail!")

✅ Connected to Gmail!


In [ ]:
profile = gmail_service.users().getProfile(
    userId="me"
).execute()

print(profile)

{'emailAddress': 'dalalabdullatif62@gmail.com', 'messagesTotal': 2958, 'threadsTotal': 2863, 'historyId': '309194'}


In [ ]:
import base64

def list_unread_emails(max_results=5):
    """
    Return the latest unread emails.
    """

    try:

        results = gmail_service.users().messages().list(
            userId="me",
            labelIds=["UNREAD"],
            maxResults=max_results
        ).execute()

        messages = results.get("messages", [])

        if not messages:
            return {
                "status": "success",
                "emails": []
            }

        emails = []

        for msg in messages:

            message = gmail_service.users().messages().get(
                userId="me",
                id=msg["id"]
            ).execute()

            headers = message["payload"]["headers"]

            subject = ""
            sender = ""

            for h in headers:

                if h["name"] == "Subject":
                    subject = h["value"]

                elif h["name"] == "From":
                    sender = h["value"]

            emails.append({
                "id": msg["id"],
                "subject": subject,
                "from": sender
            })

        return {
            "status": "success",
            "count": len(emails),
            "emails": emails
        }

    except Exception as e:

        return {
            "status": "error",
            "message": str(e)
        }

In [ ]:
import base64

def read_email(message_id):
    """
    Read the content of a Gmail message.
    """

    try:
        message = gmail_service.users().messages().get(
            userId="me",
            id=message_id,
            format="full"
        ).execute()

        headers = message["payload"]["headers"]

        subject = ""
        sender = ""

        for h in headers:
            if h["name"] == "Subject":
                subject = h["value"]

            elif h["name"] == "From":
                sender = h["value"]

        body = ""

        if "parts" in message["payload"]:

            for part in message["payload"]["parts"]:

                if part["mimeType"] == "text/plain":

                    data = part["body"]["data"]

                    body = base64.urlsafe_b64decode(
                        data.encode("UTF-8")
                    ).decode("UTF-8")

                    break

        elif "data" in message["payload"]["body"]:

            data = message["payload"]["body"]["data"]

            body = base64.urlsafe_b64decode(
                data.encode("UTF-8")
            ).decode("UTF-8")

        return {
            "status": "success",
            "subject": subject,
            "from": sender,
            "body": body
        }

    except Exception as e:

        return {
            "status": "error",
            "message": str(e)
        }

In [ ]:
import base64
from email.mime.text import MIMEText


def send_email(
    to: str,
    subject: str,
    body: str
):
    """
    Send an email using Gmail.
    """

    try:

        message = MIMEText(body)

        message["to"] = to
        message["subject"] = subject

        raw = base64.urlsafe_b64encode(
            message.as_bytes()
        ).decode()

        gmail_service.users().messages().send(
            userId="me",
            body={
                "raw": raw
            }
        ).execute()

        return {
            "status": "success",
            "message": "Email sent successfully."
        }

    except Exception as e:

        return {
            "status": "error",
            "message": str(e)
        }

In [ ]:
from langchain_core.tools import tool

@tool
def list_unread_emails_tool():
    """
    List unread emails.
    """
    return list_unread_emails()

In [ ]:
from langchain_core.tools import tool

@tool
def read_email_tool(message_id: str):
    """
    Read a Gmail message.
    """
    return read_email(message_id)

In [ ]:
@tool
def send_email_tool(
    to: str,
    subject: str,
    body: str
):
    """
    Send an email.
    """

    return send_email(
        to,
        subject,
        body
    )

In [ ]:
email_tools = [
    list_unread_emails_tool,
    read_email_tool,
    send_email_tool
]

# ==========================================
# Module 6 - Email Agent
# ==========================================

In [ ]:
email_agent = create_react_agent(
    model=llm,
    tools=email_tools,
    name="email_agent",
    prompt="""
You are an intelligent Gmail assistant.

You can:

- List unread emails
- Read emails
- Send emails

Always use tools.

Never invent email contents.

If you need more information,
ask the user.
"""
)

/tmp/ipykernel_891/2190363882.py:1: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  email_agent = create_react_agent(


# ==========================================
# Module 7 - Supervisor Agent
# ======================================

In [ ]:
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages

In [ ]:
#Create the Shared State
class AssistantState(TypedDict):
    messages: Annotated[list, add_messages]
    next_agent: str

In [ ]:
#Build the Supervisor Node
def supervisor_node(state: AssistantState):

    messages = state["messages"]

    prompt = f"""
You are the Supervisor Agent.

Your job is to decide which assistant should answer.

Available assistants:

1. calendar
   - meetings
   - appointments
   - schedule
   - events

2. email
   - gmail
   - email
   - send
   - read
   - inbox

Reply with ONLY ONE WORD:

calendar

or

email

User request:
{messages[-1].content}
"""

    decision = llm.invoke(prompt).content.strip().lower()

    if "calendar" in decision:
        return {"next_agent": "calendar"}

    return {"next_agent": "email"}




In [ ]:
#Calendar Node
def calendar_node(state: AssistantState):

    result = calendar_agent.invoke(
        {
            "messages": state["messages"]
        }
    )

    return {
        "messages": result["messages"]
    }

In [ ]:
#Email Node
def email_node(state: AssistantState):

    result = email_agent.invoke(
        {
            "messages": state["messages"]
        }
    )

    return {
        "messages": result["messages"]
    }

In [ ]:
#Router Function
def route_agent(state: AssistantState):

    return state["next_agent"]

In [ ]:
#Build the Graph
builder = StateGraph(AssistantState)

builder.add_node("supervisor", supervisor_node)
builder.add_node("calendar", calendar_node)
builder.add_node("email", email_node)

builder.set_entry_point("supervisor")

builder.add_conditional_edges(
    "supervisor",
    route_agent,
    {
        "calendar": "calendar",
        "email": "email"
    }
)

builder.add_edge("calendar", END)
builder.add_edge("email", END)

assistant = builder.compile()

Test the Supervisor

In [ ]:
response = assistant.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "What meetings do I have today?"
            }
        ]
    }
)

print(response["messages"][-1].content)

You have no meetings scheduled for today.


In [ ]:
response = assistant.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Show my unread emails."
            }
        ]
    }
)

print(response["messages"][-1].content)

ChatGoogleGenerativeAIError: Error calling model 'gemini-2.5-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 40.04249494s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTi

# ==========================================
# Module 8 - Memory
# ======================================

In [ ]:
from langgraph.checkpoint.memory import MemorySaver

In [ ]:
memory = MemorySaver()

In [ ]:
assistant = builder.compile()

In [ ]:
assistant = builder.compile(
    checkpointer=memory
)

In [ ]:
config = {
    "configurable": {
        "thread_id": "dalal_session"
    }
}

In [ ]:
assistant.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Show my unread emails."
            }
        ]
    },
    config=config
)

ChatGoogleGenerativeAIError: Error calling model 'gemini-2.5-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 7.512377011s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTi

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver

class MemoryState(TypedDict):
    messages: list

In [ ]:
def remember_node(state: MemoryState):

    print("Messages currently stored:")
    print(state["messages"])

    return state

In [ ]:
builder = StateGraph(MemoryState)

builder.add_node("remember", remember_node)

builder.set_entry_point("remember")

builder.add_edge("remember", END)

memory = MemorySaver()

memory_graph = builder.compile(
    checkpointer=memory
)

In [ ]:
config = {
    "configurable": {
        "thread_id": "dalal_memory"
    }
}

In [ ]:
memory_graph.invoke(
    {
        "messages": [
            "Create meeting tomorrow"
        ]
    },
    config=config
)

Messages currently stored:
['Create meeting tomorrow']


{'messages': ['Create meeting tomorrow']}

In [ ]:
memory_graph.invoke(
    {
        "messages": [
            "Move it to 3 PM"
        ]
    },
    config=config
)

Messages currently stored:
['Move it to 3 PM']


{'messages': ['Move it to 3 PM']}

# ==========================================
# Module 9 - Human Approval
# ======================================

In [ ]:
pending_action = None

def request_confirmation(action_type, action_data):
    global pending_action

    pending_action = {
        "type": action_type,
        "data": action_data
    }

    if action_type == "delete_calendar":
        return f"""
⚠️ Confirmation Required

You are about to delete this calendar event:

📅 {action_data['title']}

Do you want to continue?

Reply:
YES
or
NO
"""

    elif action_type == "send_email":
        return f"""
⚠️ Confirmation Required

You are about to send an email.

To: {action_data['to']}
Subject: {action_data['subject']}

Reply:
YES
or
NO
"""

    return "Confirmation required."

In [ ]:
def execute_pending_action(user_response):

    global pending_action

    if pending_action is None:
        return "No pending action."

    if user_response.strip().lower() != "yes":
        pending_action = None
        return "Action cancelled."

    action = pending_action
    pending_action = None

    if action["type"] == "delete_calendar":

        return delete_calendar_event(
            action["data"]["event_id"]
        )

    elif action["type"] == "send_email":

        data = action["data"]

        return send_email(
            data["to"],
            data["subject"],
            data["body"]
        )

    return "Unknown action."